# 06 RQ3 Multi-Modal Fusion and Omega

Synopsis RQ3: compare spatial-only, news-only, and fused feature sets on stress-window Omega proxy association / ranking.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from project_paths import load_env, ensure_output_dirs
ROOT = load_env()
PATHS = ensure_output_dirs(ROOT)
print("ROOT", ROOT)
print({k: str(v) for k, v in PATHS.items()})

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, f1_score, roc_auc_score, confusion_matrix, RocCurveDisplay
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
sns.set_theme(style="whitegrid", context="talk")
df = pd.read_csv(PATHS["processed"] / "daily_panel_clean.csv", parse_dates=["date"]).sort_values("date").reset_index(drop=True)
print(df.shape)


In [ ]:
from sklearn.linear_model import LinearRegression

spatial = [c for c in ["port_congestion_hours","chokepoint_vessel_count","vessel_draught_anomaly"] if c in df.columns]
news = [c for c in ["news_event_impact_score","finbert_risk_probability"] if c in df.columns]
work = df.dropna(subset=["omega_ratio_proxy"]).copy()
# Stress window: top quartile of WTI realized vol
vol = work["wti_spot"].pct_change().rolling(10).std()
work = work.assign(stress=(vol >= vol.quantile(0.75)))
stress = work[work["stress"]].dropna(subset=spatial + news + ["omega_ratio_proxy"])

split = int(len(work) * 0.7)
results = []
preds_hold = {}
for name, feats in [
    ("spatial_only", spatial),
    ("news_only", news),
    ("early_fusion", spatial + news),
]:
    feats = [f for f in feats if f in work.columns]
    if len(feats) < 1:
        continue
    d = work.dropna(subset=feats + ["omega_ratio_proxy"]).copy()
    tr, te = d.iloc[:int(len(d)*0.7)], d.iloc[int(len(d)*0.7):]
    model = LinearRegression().fit(tr[feats], tr["omega_ratio_proxy"])
    pred = model.predict(te[feats])
    rmse = mean_squared_error(te["omega_ratio_proxy"], pred) ** 0.5
    # Omega-style score: mean predicted omega in stress days within test window
    te_stress = te.copy()
    te_stress["pred"] = pred
    vol_te = te_stress["wti_spot"].pct_change().rolling(10).std()
    thr = vol_te.quantile(0.75)
    omega_stress = te_stress.loc[vol_te >= thr, "pred"].mean()
    results.append({
        "Model": name,
        "Feature Set": ",".join(feats),
        "Validation Approach": "chronological 70/30",
        "RMSE_omega": rmse,
        "MeanPredOmega_stress": omega_stress,
        "Key Takeaway": "",
    })
    preds_hold[name] = (te["date"], te["omega_ratio_proxy"], pred)

metrics = pd.DataFrame(results)
if len(metrics):
    best = metrics.sort_values("RMSE_omega").iloc[0]["Model"]
    metrics.loc[metrics["Model"]==best, "Key Takeaway"] = "Lowest Omega RMSE; check stress mean too"
metrics.to_csv(PATHS["outputs"] / "metrics_rq3.csv", index=False)
metrics.to_csv(PATHS["tables"] / "model_performance_rq3.csv", index=False)
metrics


### Figure 6. RQ3 modality comparison


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
plot_df = metrics.set_index("Model")[["RMSE_omega"]]
plot_df.plot(kind="bar", ax=ax, legend=False, color="#2a9d8f")
ax.set_title("Figure 6. RQ3 Omega RMSE by modality")
ax.set_ylabel("RMSE")
ax.set_xlabel("Model")
fig.tight_layout()
fig.savefig(PATHS["figures"] / "fig06_rq3_modality.png", dpi=200)
plt.show()
